# Pointer Networks (Ptr-Net) — Theory & Mathematical Foundations

> **Note:** This notebook is theory-only (all markdown, no code). For the full implementation, training loop, and experiments, see [`notebook_implementation.ipynb`](./notebook_implementation.ipynb).

## 1. Definition

### Origin

Pointer Networks were introduced by **Vinyals, Fortunato, and Jaitly** at NeurIPS 2015 in the paper *"Pointer Networks"*. The model was motivated by combinatorial optimisation problems — in particular, the **Travelling Salesman Problem (TSP)**, the **convex hull problem**, and the **Delaunay triangulation problem** — where the output is a permutation of the input elements.

### Category

| Attribute | Value |
|-----------|-------|
| Learning paradigm | Supervised deep learning |
| Model family | Sequence-to-sequence (seq2seq) |
| Core mechanism | Attention-based pointer over input elements |
| Training signal | Optimal tour labels (ground truth permutations) |

### Key Concepts

- **Pointer mechanism:** At each decoding step the model "points" directly to one of the input elements (cities), rather than generating a token from a fixed vocabulary. This makes the output vocabulary *dynamic* — it depends on the input itself.
- **Attention as a soft pointer:** The attention scores over encoder hidden states are interpreted as a probability distribution over input positions. The city with the highest probability is selected (pointed to) at each step.

### Why Was It Created?

Standard seq2seq models (e.g., the original encoder–decoder of Sutskever et al. 2014) output tokens from a **fixed vocabulary** whose size is determined at training time. This is fundamentally incompatible with problems where:

1. The output is a **permutation of the input** (a set of cities to visit).
2. The **output length equals the input length**, which itself varies between instances.

For TSP with $n$ cities, the output vocabulary must contain exactly those $n$ cities — a vocabulary that changes with every new problem instance. A classical seq2seq model would need to be retrained for every different $n$. The Pointer Network solves this by making the output distribution a function of the *encoder hidden states* rather than a fixed embedding matrix, so the model naturally generalises across different input sizes.

## 2. Formal Description

### Input and Output

Let the input be a sequence of $n$ city coordinates:

$$X = \{x_1, x_2, \ldots, x_n\}, \quad x_i \in \mathbb{R}^2$$

where:
- $X$ — set of $n$ input city coordinates
- $x_i$ — 2D coordinates of city $i$, normalised to $[0,1]^2$
- $n$ — number of cities in the instance

The desired output is a **permutation** $\pi$ of $\{1, 2, \ldots, n\}$ representing the order in which cities are visited:

$$\pi = (\pi_1, \pi_2, \ldots, \pi_n)$$

where:
- $\pi$ — output tour permutation of city indices
- $\pi_t$ — city selected at decoding step $t$

### Probability Factorisation

The joint probability of a tour $\pi$ given input $X$ is factorised autoregressively using the chain rule:

$$P(\pi \mid X) = \prod_{t=1}^{n} P(\pi_t \mid \pi_1, \pi_2, \ldots, \pi_{t-1},\, X)$$

where:
- $P(\pi \mid X)$ — joint probability of the full tour given the input
- each factor — conditional distribution over remaining unvisited cities at step $t$

### Encoder

An LSTM encoder reads the input sequence and produces a hidden state for each city:

$$h_i = \text{LSTM}_{\text{enc}}(x_i, h_{i-1}), \quad i = 1, \ldots, n$$

where:
- $h_i \in \mathbb{R}^d$ — encoder hidden state for city $i$
- $d$ — LSTM hidden dimension
- $h_0$ — zero-initialised initial hidden state

### Decoder

An LSTM decoder maintains a state $s_t \in \mathbb{R}^d$ that is updated at each step:

$$s_t = \text{LSTM}_{\text{dec}}(\tilde{x}_{\pi_{t-1}},\, s_{t-1})$$

where:
- $s_t$ — decoder hidden state at step $t$
- $\tilde{x}_{\pi_{t-1}} = W_{\text{city}}\,x_{\pi_{t-1}}$ — embedding of the previously selected city
- $W_{\text{city}}$ — shared city embedding matrix

### Attention / Pointer Scores

At decoding step $t$, an attention score is computed for each encoder position $i$:

$$u_i^t = v^\top \tanh\!\left(W_1 \cdot h_i + W_2 \cdot s_t\right)$$

where:
- $u_i^t \in \mathbb{R}$ — unnormalised pointer score for city $i$ at step $t$
- $W_1, W_2 \in \mathbb{R}^{d \times d}$ — learned weight matrices projecting encoder and decoder states
- $v \in \mathbb{R}^d$ — learned query vector; dot product extracts a scalar score

### Pointer Distribution with Masking

Visited cities must not be selected again. Let $\mathcal{V}_t$ denote the set of already-visited city indices at step $t$. The masked scores are:

$$\tilde{u}_i^t = \begin{cases} u_i^t & \text{if } i \notin \mathcal{V}_t \\ -\infty & \text{if } i \in \mathcal{V}_t \end{cases}$$

where:
- $\tilde{u}_i^t$ — masked score for city $i$ at step $t$
- $\mathcal{V}_t$ — set of already-visited city indices at step $t$

The pointer probability distribution is then:

$$P(\pi_t = i \mid \pi_1, \ldots, \pi_{t-1},\, X) = \text{softmax}(\tilde{u}^t)_i = \frac{\exp(\tilde{u}_i^t)}{\sum_{j=1}^{n} \exp(\tilde{u}_j^t)}$$

where:
- $P(\pi_t = i \mid \ldots)$ — probability of pointing to city $i$ at step $t$, given past choices and input
- $\tilde{u}^t$ — vector of masked scores for all cities at step $t$

Because $\exp(-\infty) = 0$, visited cities receive zero probability.

### Training Loss

Given a training example $(X, \pi^*)$ where $\pi^*$ is the **optimal tour** (ground-truth label), the supervised training loss is the negative log-likelihood (NLL) summed over all decoding steps:

$$\mathcal{L}_{\text{NLL}} = -\sum_{t=1}^{n} \log P(\pi_t^* \mid \pi_1^*, \ldots, \pi_{t-1}^*,\, X)$$

where:
- $\mathcal{L}_{\text{NLL}}$ — negative log-likelihood loss, minimised during training
- $\pi^*$ — ground-truth tour (optimal / nn2opt label)
- $\pi_t^*$ — target city at step $t$ under teacher forcing

Minimising $\mathcal{L}_{\text{NLL}}$ over a dataset of $(X, \pi^*)$ pairs trains the model to reproduce optimal tours via teacher forcing.

## 3. Architecture / Algorithm Steps

### Encoder

The encoder is a **bidirectional or unidirectional LSTM** that processes the sequence of city coordinates one by one:

1. Each city coordinate $x_i \in \mathbb{R}^2$ is (optionally) projected to a higher-dimensional embedding.
2. The LSTM reads the sequence left-to-right, producing hidden states $h_1, h_2, \ldots, h_n$.
3. These hidden states form the **memory bank** from which the decoder will point.

The final encoder hidden state $(h_n, c_n)$ initialises the decoder.

### Decoder

The decoder runs for exactly $n$ steps (one per city to select):

1. At step $t$, the decoder LSTM receives the embedding of the **previously chosen city** $x_{\pi_{t-1}}$ as input and updates its hidden state $s_t$.
2. Attention scores $u_i^t$ are computed between $s_t$ and every encoder state $h_i$.
3. Visited cities are masked (set to $-\infty$).
4. A softmax produces the pointer distribution; the city $\pi_t = \arg\max_i P(\pi_t = i \mid \ldots)$ is selected (greedy) or a beam is maintained.
5. The selected city is added to $\mathcal{V}_t$ and the process repeats.

### Masking

A binary mask vector $m^t \in \{0, 1\}^n$ tracks visited cities. Before applying softmax, scores of visited positions are replaced with $-\infty$:

$$\tilde{u}_i^t = u_i^t + (1 - m_i^t) \cdot (-\infty)$$

This guarantees that no city is selected twice, enforcing the permutation constraint without any additional architectural change.

### Greedy vs Beam Search Decoding

| Strategy | Description | Quality | Speed |
|----------|-------------|---------|-------|
| **Greedy** | Select $\arg\max$ at every step | Lower | Fastest |
| **Sampling** | Sample from the distribution; repeat $k$ times, keep best | Medium | Medium |
| **Beam search** | Maintain top-$B$ partial sequences at each step | Higher | Slowest (×B) |

In practice, greedy decoding is used for inference speed; beam search with small $B$ (e.g., $B=10$) yields noticeably better tours.

### Pipeline Diagram

```
INPUT CITIES
x1   x2   x3  ...  xn
 |    |    |         |
[LSTM encoder — left to right]
 |    |    |         |
h1   h2   h3  ...  hn    <-- encoder memory bank
                    |
              (h_n, c_n) --> initialise decoder

DECODER (step t=1,2,...,n)

  x_{pi_{t-1}} --> [LSTM decoder] --> s_t
                        |
             [Attention: u_i^t = v^T tanh(W1*h_i + W2*s_t)]
                        |
                  [Apply mask]
                        |
                   [Softmax]
                        |
               P(pi_t | ...)  -->  select pi_t  --> add to visited

OUTPUT TOUR: (pi_1, pi_2, ..., pi_n)
```

## 4. Complexity Analysis

### Time Complexity

Let $n$ be the number of cities and $d$ the LSTM hidden dimension.

| Phase | Operation | Cost |
|-------|-----------|------|
| Encoding | $n$ LSTM steps, each $O(d^2)$ | $O(n \cdot d^2)$ |
| Decoding | $n$ decoder LSTM steps, each $O(d^2)$ | $O(n \cdot d^2)$ |
| Attention | At each of $n$ decoder steps: $n$ dot products of size $d$ | $O(n^2 \cdot d)$ |
| **Total** | Dominated by attention | $O(n^2 \cdot d)$ |

The **dominant term** for a forward pass is $O(n^2 \cdot d)$: for each of the $n$ decoder steps, attention scores must be computed over all $n$ encoder states.

### Space Complexity

The encoder hidden states must all be stored in memory to allow attention at every decoder step:

$$\text{Space} = O(n \cdot d)$$

where:
- $n$ — number of cities (input length)
- $d$ — LSTM hidden dimension

The decoder state at any given time is a single vector $s_t \in \mathbb{R}^d$, adding only $O(d)$.

### Comparison with GNN

A Graph Neural Network (GNN) with $L$ message-passing layers on a complete graph of $n$ nodes has:

$$\text{Time}_{\text{GNN}} = O(L \cdot n^2 \cdot d)$$

where:
- $L$ — number of GNN message-passing layers
- $n^2$ — all pairwise edges in the complete graph
- $d$ — node/edge embedding dimension

Both Ptr-Net and GNN scale quadratically in $n$ due to pairwise interactions. The GNN constant factor is $L$ (number of layers), while Ptr-Net's constant factor is $1$ (single-pass encoder).

### Parallelisation

A critical limitation of Ptr-Net is that the **LSTM decoder is sequential**: step $t$ depends on step $t-1$, so decoding cannot be parallelised across time steps. This contrasts with Transformer-based models (e.g., the Attention Model of Kool et al. 2019), where the encoder is fully parallel and the decoder can be batched more efficiently on modern hardware.

$$\text{Sequential decoder steps: } t = 1 \to 2 \to \cdots \to n \quad (\text{not parallelisable})$$

## 5. Strengths and Limitations

| | Strengths | Limitations |
|---|-----------|-------------|
| **Variable input size** | Handles any $n$ at inference time — the pointer mechanism adapts to the current input length without architectural changes | Generalisation to $n$ much larger than training $n$ degrades significantly in practice |
| **Lightweight** | Fewer parameters than graph-based or Transformer models; fast training on small $n$ | LSTM sequential decoding is slow at inference for large $n$ |
| **Interpretable attention** | The pointer distribution $P(\pi_t = i \mid \ldots)$ is directly interpretable: it shows which city the model is "considering" at each step | Attention weights may be noisy and do not guarantee correctness |
| **Exact permutation output** | The masking mechanism strictly enforces a valid permutation — no city is visited twice | Masking only prevents revisits; it does not encode richer constraints (e.g., time windows) without modification |
| **Supervised training** | Clean, well-understood training objective (cross-entropy on optimal tours) | Requires optimal tour labels for every training instance; exact optimal labels are only tractable for small $n$ (brute-force: $n \leq 10$; heuristic labels: up to $n \approx 100$) |
| **Baseline model** | Simple architecture, easy to implement and debug; good starting point before more complex models | Superseded by Transformer AM (Kool et al. 2019) and other modern approaches on solution quality |

## 6. Use Case Explanation

### When to Use Pointer Networks

- **Small-to-medium TSP instances ($n \leq 100$):** Ptr-Net was designed and validated on this range. For $n \leq 50$ it achieves near-optimal tours with beam search when trained on comparable instance sizes.
- **Interpretability of the selection process matters:** Because the pointer distribution is a proper probability over input cities at every step, it is possible to inspect and visualise what the model is attending to. This can be useful for debugging or explaining decisions in an academic or research context.
- **As a baseline before trying more complex models:** Ptr-Net is the historical predecessor of modern neural combinatorial optimisation methods. Implementing it first gives a clear understanding of the encoder–decoder–pointer paradigm, making it easier to understand and extend to the Transformer Attention Model.
- **Limited compute environments:** Ptr-Net is lightweight and can be trained on a single GPU in a few hours for $n \leq 50$.

### When NOT to Use Pointer Networks

- **Large instances ($n \gg 100$):** The $O(n^2 \cdot d)$ sequential decoder becomes a bottleneck, and the model's generalisation across sizes degrades sharply.
- **Time-window constraints (TSPTW) without modification:** The basic Ptr-Net architecture has no mechanism to encode time windows or feasibility constraints. Incorporating such constraints requires significant architectural modifications (e.g., infeasibility masking, constraint-aware embeddings).
- **When state-of-the-art solution quality is required:** For competitive results, the Transformer Attention Model (Kool et al. 2019) or POMO (Kwon et al. 2020) should be preferred.
- **Reinforcement learning training:** While RL-trained Ptr-Net exists (Bello et al. 2016), the architecture is less suited for RL than Transformer-based models, which benefit more from the parallelism during policy gradient estimation.

## 10. Experimental Analysis

*This section provides a theoretical discussion of expected experimental behaviour, complementing the empirical results in `notebook_implementation.ipynb`.*

### Generalisation Gap

One of the most well-documented limitations of Ptr-Net is its **poor out-of-distribution generalisation** with respect to instance size. A model trained exclusively on TSP instances with $n = 10$ cities exhibits rapid performance degradation when evaluated on instances with $n = 50$ or $n = 100$:

- The LSTM encoder produces hidden states whose scale and structure reflect the training distribution. When presented with more cities, the attention scores become diffuse and less discriminative.
- The positional encoding implicit in the LSTM ordering does not transfer to larger sequences.
- Empirically, the **optimality gap** (percentage above optimal tour length) can increase from $\approx 1\%$ at $n=10$ to $> 20\%$ at $n=50$ without retraining.

This motivates training separate models for different size regimes or using data augmentation strategies.

### Comparison: Ptr-Net vs GNN

| Aspect | Pointer Network | Graph Neural Network |
|--------|----------------|---------------------|
| Input representation | Ordered sequence of coordinates | Graph: nodes (cities) + edges (distances) |
| Structural inductive bias | Sequential (LSTM processes input in order) | Relational (message passing over graph edges) |
| Permutation invariance | No — input order affects output | Yes (with appropriate pooling) |
| Distance encoding | Implicit via coordinate embeddings | Explicit edge features |
| Best suited for | TSP as a sequence-generation task | TSP as a graph optimisation task |

GNNs explicitly model the **graph structure** of TSP (pairwise distances as edge features), which gives them a richer inductive bias for routing problems. Ptr-Net treats TSP as a sequence-to-sequence problem, ignoring the complete graph structure and relying on coordinate embeddings alone.

### Comparison: Ptr-Net vs Transformer Attention Model (Kool et al. 2019)

The Transformer Attention Model (AM) is the direct successor to Ptr-Net. It replaces the LSTM encoder and decoder with multi-head self-attention:

| Aspect | Ptr-Net (Vinyals 2015) | Transformer AM (Kool 2019) |
|--------|----------------------|---------------------------|
| Encoder | Unidirectional LSTM | $L$-layer Transformer with multi-head attention |
| Decoder | LSTM + additive attention | Single-layer multi-head attention (context query) |
| Parallelism | Sequential encoder & decoder | Fully parallel encoder; faster decoder |
| Training | Supervised (teacher forcing) | Reinforcement learning (REINFORCE with baseline) |
| No optimal labels needed | No (needs optimal $\pi^*$) | Yes (self-supervised via RL reward = tour length) |
| Relationship | Original pointer mechanism | Direct architectural evolution of Ptr-Net |

The AM retains the **pointer mechanism** concept (soft attention over input positions) but achieves significantly better performance by: (1) using richer contextual embeddings from the Transformer encoder, (2) removing the need for optimal tour labels via RL training, and (3) enabling better parallelisation.

### Performance Comparison on TSP $n = 100$

The following table summarises approximate **optimality gaps** (percentage above the Concorde exact solver) reported in the literature for TSP with $n = 100$ cities:

| Model | Decoding | Approx. Gap to Optimal (TSP $n=100$) | Training Paradigm |
|-------|----------|--------------------------------------|------------------|
| **Ptr-Net** (Vinyals et al. 2015) | Beam search $B=10$ | $\approx 8 - 10\%$ | Supervised |
| **GNN + RL** (e.g., S2V-DQN) | Greedy | $\approx 3 - 5\%$ | Reinforcement Learning |
| **Transformer AM** (Kool et al. 2019) | Greedy | $\approx 1.2\%$ | Reinforcement Learning |
| **Transformer AM** (Kool et al. 2019) | Sampling $\times 1280$ | $\approx 0.5\%$ | Reinforcement Learning |
| **Concorde** (exact solver) | — | $0\%$ (optimal) | — |

> **Note:** These figures are indicative and vary by dataset and implementation. The key takeaway is the consistent improvement from Ptr-Net $\to$ GNN $\to$ Transformer AM as architectural expressiveness and training paradigms improve.

## 11. References

| # | Reference | Key Contribution |
|---|-----------|------------------|
| 1 | **Vinyals, O., Fortunato, M., & Jaitly, N. (2015).** *Pointer Networks.* Advances in Neural Information Processing Systems (NeurIPS), 28. | Original Pointer Network architecture; supervised training on TSP, convex hull, Delaunay triangulation. |
| 2 | **Bello, I., Pham, H., Le, Q. V., Norouzi, M., & Bengio, S. (2016).** *Neural Combinatorial Optimization with Reinforcement Learning.* ICLR 2017 Workshop. | First application of reinforcement learning (REINFORCE / actor-critic) to train a Ptr-Net for TSP, eliminating the need for optimal tour labels. |
| 3 | **Kool, W., van Hoof, H., & Welling, M. (2019).** *Attention, Learn to Solve Routing Problems!* International Conference on Learning Representations (ICLR). | Transformer-based Attention Model (AM); direct successor to Ptr-Net. Replaces LSTM with multi-head self-attention, trains with REINFORCE + greedy baseline. State-of-the-art neural solver for TSP and VRP at time of publication. |